# 04 Seq2Seq 与 Teacher Forcing

这里不训练大型翻译模型，只把 Decoder 最容易混淆的“输入和目标错开一位”彻底看清。


In [ ]:
import torch

BOS_ID = 1
EOS_ID = 2
PAD_ID = 0

# 假设真实目标序列是：
# <bos> I love NLP <eos>
target_full = torch.tensor([
    [BOS_ID, 10, 11, 12, EOS_ID],
    [BOS_ID, 20, 21, EOS_ID, PAD_ID],
])

decoder_input = target_full[:, :-1]
decoder_target = target_full[:, 1:]

print("target_full:")
print(target_full)

print("\ndecoder_input:")
print(decoder_input)

print("\ndecoder_target:")
print(decoder_target)

训练时：

```text
输入： <bos> I    love NLP
目标： I     love NLP  <eos>
```

这就是最基本的 Teacher Forcing 组织方式。


## Padding 位置不能计算损失

In [ ]:
from torch import nn

batch_size, target_length = decoder_target.shape
vocab_size = 30

logits = torch.randn(
    batch_size,
    target_length,
    vocab_size,
)

loss_fn = nn.CrossEntropyLoss(
    ignore_index=PAD_ID,
)

loss = loss_fn(
    logits.reshape(-1, vocab_size),
    decoder_target.reshape(-1),
)

print("masked cross entropy:", float(loss))

## 推理阶段为什么不同？

推理时没有真实 `decoder_target`，因此：

```text
<bos>
→ 预测 token_1
→ 把 token_1 作为下一步输入
→ 预测 token_2
→ ...
```

这为后续自回归 Transformer Decoder 打基础。
